## Load data from Bronze


In [0]:
historical_stratify_df_spark = spark.read.table("predictivemaintenancecatalog.bronze_data_not_quality_checked.historical_stratified")
display(historical_stratify_df_spark)


## Quality Checks

In [0]:
# check for null values
from pyspark.sql import functions as F
historical_stratify_df_spark.select([
    F.count(F.when(F.col(c).isNull(), 1)).alias(c)
    for c in historical_stratify_df_spark.columns
]).show()

historical_stratify_df_spark = historical_stratify_df_spark.dropna()

historical_stratify_df_spark.select([
    F.count(F.when(F.col(c).isNull(), 1)).alias(c)
    for c in historical_stratify_df_spark.columns
]).show()

In [0]:
historical_stratify_df_spark.dropDuplicates(["UDI"])

In [0]:
#No duplicate UDI identifiers
before = historical_stratify_df_spark.count()
after = historical_stratify_df_spark.dropDuplicates(["UDI"]).count()
print(f"Before: {before}, After: {after}, Duplicates: {before - after}")
dup=before-after
if dup>0:
     historical_stratify_df_spark=historical_stratify_df_spark.dropDuplicates(["UDI"])



In [0]:
historical_stratify_df_spark.columns 

In [0]:
numerical_cols= [ 'Air_temperature_K','Process_temperature_K', 'Rotational_speed_rpm', 'Torque_Nm', 'Tool_wear_min',]
catergorical_cols = ['Product ID', 'Type',]

In [0]:
# check to make sure that numerical columns are greater than 0
for col in numerical_cols:
    historical_stratify_df_spark = historical_stratify_df_spark.filter(F.col(col) >= 0)
    historical_stratify_df_spark.filter(F.col(col) < 0).show()

## Write the historical data to the silver 

In [0]:
spark.sql("Create schema if not exists predictivemaintenancecatalog.silver_data")

clean_cols = [c.replace(" ", "_").replace("[", "").replace("]", "") for c in historical_stratify_df_spark.columns]
historical_stratify_df_spark.toDF(*clean_cols).write.format("delta").mode("overwrite").saveAsTable("predictivemaintenancecatalog.silver_data.historical_stratified")